# HALO v2 — Análisis Exploratorio de Datos (EDA Avanzado)
> **Taller de Sistemas Inteligentes** | Sprint 0 — La Paz, Bolivia

Analiza los **10,000 incidentes sintéticos** generados para validar patrones urbanos reales.

### Fundamentación Académica de los Datos (No Aleatorios)
Los datos generados se fundamentan en la simulación de fenómenos urbanos reales bajo la técnica de **Monte Carlo con pesos demográficos**:
1. **Pesos Poblacionales (INE/GAMLP)**: La probabilidad de un incidente respeta la densidad poblacional real (ej. Max Paredes 25%, Mallasa 5%).
2. **Estacionalidad Diaria (Cronobiología Urbana)**: Distribución bimodal (picos a las 12h y 20h), coincidiendo con estudios de victimización.
3. **Estacionalidad Semanal (Ocio)**: Viernes y sábados tienen multiplicadores positivos de riesgo (+40% a +50%).

*Propósito: Esta generación parametrizada nos permite evaluar si los modelos IA en el Sprint 1 son capaces de 'redescubrir' matemáticamente las reglas urbanas ocultas en el dataset.*

In [ ]:
import os, sys, warnings
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

# Estilo global premium
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444',
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'grid.color': '#333',
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'figure.titlesize': 16,
})

# Rutas
NB_DIR = os.path.dirname(os.path.abspath('__file__'))
ROOT_DIR = os.path.join(NB_DIR, '..')
RAW_DAILY = os.path.join(ROOT_DIR, 'data', 'raw', 'synthetic_lapaz_daily.csv')
RAW_IND   = os.path.join(ROOT_DIR, 'data', 'raw', 'synthetic_lapaz_v1.csv')
OUT_DIR   = os.path.join(ROOT_DIR, 'output', 'eda_plots')
os.makedirs(OUT_DIR, exist_ok=True)

DIAS_SEMANA = ['Lun', 'Mar', 'Mie', 'Jue', 'Vie', 'Sab', 'Dom']
PALETA_ZONAS = ['#e63946','#457b9d','#2a9d8f','#e9c46a','#f4a261','#264653','#8ecae6']

print('Librerias listas.')

## 1. Carga de Datos

In [ ]:
df_daily = pd.read_csv(RAW_DAILY)
df_ind   = pd.read_csv(RAW_IND)

# Agregar columna de dia de semana
df_daily['weekday'] = [datetime.strptime(x, '%Y-%m-%d').weekday() for x in df_daily['ds']]
df_daily['weekday_name'] = [DIAS_SEMANA[w] for w in df_daily['weekday']]
df_daily['mes'] = [int(x[5:7]) for x in df_daily['ds']]
df_daily['anio'] = [int(x[:4]) for x in df_daily['ds']]

print(f'Dataset diario : {len(df_daily):,} registros')
print(f'Dataset individual: {len(df_ind):,} incidentes')
print(f'Periodo: {df_daily["ds"].min()} a {df_daily["ds"].max()}')
df_daily.head()

## 2. Estadísticas Descriptivas por Macrodistrito

In [ ]:
stats = df_daily.groupby('macrodistrito')['y'].agg(
    Dias_Datos='count',
    Media='mean',
    Mediana='median',
    Std='std',
    Min='min',
    Max='max',
    P25=lambda x: x.quantile(0.25),
    P75=lambda x: x.quantile(0.75)
).round(2).sort_values('Media', ascending=False)
stats

## 3. Serie de Tiempo — Media Móvil 7 días por Zona

In [ ]:
zonas = sorted(df_daily['macrodistrito'].unique())
n = len(zonas)

fig, ax = plt.subplots(figsize=(16, 6))
for i, zona in enumerate(zonas):
    g = df_daily[df_daily['macrodistrito'] == zona].set_index('ds')['y']
    smooth = g.rolling(7, min_periods=1).mean()
    xs = [datetime.strptime(x, '%Y-%m-%d') for x in smooth.index]
    ax.plot(xs, smooth.values, label=zona, color=PALETA_ZONAS[i], linewidth=1.6)

ax.set_title('Serie de Tiempo de Incidentes — La Paz (Media Movil 7 dias)', color='white')
ax.set_xlabel('Fecha'); ax.set_ylabel('Incidentes/dia')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', framealpha=0.3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '01_series_tiempo.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Mapa de Calor de Hotspots — La Paz (Lat/Lon)

In [ ]:
# Coordenadas de referencia por macrodistrito (fronteras aproximadas de La Paz)
BBOX = {'lat_min': -16.62, 'lat_max': -16.45, 'lon_min': -68.22, 'lon_max': -68.05}

fig, ax = plt.subplots(figsize=(12, 10))

# Fondo oscuro simulando mapa
ax.set_facecolor('#0a1628')

# Hexbin density map de incidentes
hb = ax.hexbin(
    df_ind['longitude'], df_ind['latitude'],
    gridsize=35, cmap='YlOrRd', alpha=0.85, mincnt=1
)
cb = fig.colorbar(hb, ax=ax, label='Densidad de incidentes')
cb.ax.yaxis.label.set_color('white')
cb.ax.tick_params(colors='white')

# Etiquetas de macrodistritos
centroides = {
    'Max Paredes': (-16.495, -68.13),
    'Centro':      (-16.505, -68.115),
    'Periferica':  (-16.475, -68.10),
    'Cotahuma':    (-16.520, -68.15),
    'San Antonio': (-16.535, -68.09),
    'Sur':         (-16.56,  -68.11),
    'Mallasa':     (-16.585, -68.08),
}
for nombre, (lat, lon) in centroides.items():
    ax.annotate(nombre, xy=(lon, lat), fontsize=9, color='white',
                ha='center', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc='#0a1628', alpha=0.6))

ax.set_xlim(BBOX['lon_min'], BBOX['lon_max'])
ax.set_ylim(BBOX['lat_min'], BBOX['lat_max'])
ax.set_title('Mapa de Densidad de Incidentes — La Paz, Bolivia\n(Hexbin: cada celda = densidad acumulada 2023-2026)',
             color='white', pad=15)
ax.set_xlabel('Longitud', color='white')
ax.set_ylabel('Latitud', color='white')
ax.tick_params(colors='white')
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '04_mapa_hotspots.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Zonas mas calientes: norte (Max Paredes / Centro) segun distribucion demografica.')

## 5. Patron Semanal por Macrodistrito

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel izquierdo: promedio global por dia de semana
means_global = df_daily.groupby('weekday_name')['y'].mean().reindex(DIAS_SEMANA)
bars = axes[0].bar(means_global.index, means_global.values,
                   color=PALETA_ZONAS, edgecolor='none', width=0.7)
axes[0].set_title('Promedio Global de Incidentes por Dia de Semana')
axes[0].set_ylabel('Promedio incidentes/dia')
axes[0].set_xlabel('')
axes[0].grid(axis='y', alpha=0.3)
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.2f}',
                 ha='center', va='bottom', fontsize=9, color='white')

# Panel derecho: boxplot por zona
data_box = [df_daily[df_daily['macrodistrito'] == z]['y'].values for z in zonas]
bp = axes[1].boxplot(data_box, labels=zonas, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], PALETA_ZONAS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for element in ['whiskers', 'caps', 'medians', 'fliers']:
    for item in bp[element]:
        item.set_color('white')
axes[1].set_title('Distribucion de Incidentes por Macrodistrito')
axes[1].set_xticklabels(zonas, rotation=30, ha='right')
axes[1].set_ylabel('Incidentes/dia')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '02_patron_semanal.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Heatmap: Macrodistrito vs Dia de Semana

In [ ]:
pivot = df_daily.groupby(['macrodistrito', 'weekday_name'])['y'].mean().unstack()[DIAS_SEMANA]

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='magma', linewidths=0.5,
            linecolor='#0f0f1a', ax=ax,
            cbar_kws={'label': 'Promedio incidentes/dia'},
            annot_kws={'size': 10, 'color': 'white'})
ax.set_title('Mapa de Calor: Macrodistrito x Dia de Semana', pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', colors='white')
ax.tick_params(axis='y', colors='white', rotation=0)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '03_heatmap_zona_dia.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Distribucion por Categoria y Severidad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Categorias
cat_counts = df_ind['categoria'].value_counts()
colors_cat = sns.color_palette('husl', len(cat_counts))
wedges, texts, autotexts = axes[0].pie(
    cat_counts.values, labels=cat_counts.index,
    autopct='%1.1f%%', colors=colors_cat,
    startangle=90, pctdistance=0.8,
    wedgeprops=dict(edgecolor='#0f0f1a', linewidth=1.5)
)
for text in texts: text.set_color('white')
for at in autotexts: at.set_color('white'); at.set_fontsize(9)
axes[0].set_title('Distribucion por Categoria de Incidente')

# Severidad (1-5)
sev_counts = df_ind['severidad'].value_counts().sort_index()
axes[1].bar(sev_counts.index, sev_counts.values,
            color=['#2a9d8f','#57cc99','#e9c46a','#f4a261','#e63946'])
axes[1].set_title('Distribucion de Severidad de Incidentes (1=Bajo, 5=Critico)')
axes[1].set_xlabel('Nivel de Severidad')
axes[1].set_ylabel('Cantidad de incidentes')
axes[1].grid(axis='y', alpha=0.3)
for i, (idx, val) in enumerate(sev_counts.items()):
    axes[1].text(idx, val + 20, f'{val:,}', ha='center', fontsize=10, color='white')

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '05_categoria_severidad.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Tendencia Anual — Evolucion del Volumen de Incidentes

In [ ]:
monthly = df_daily.copy()
monthly['periodo'] = [x[:7] for x in monthly['ds']]  # YYYY-MM
monthly_total = monthly.groupby(['periodo', 'macrodistrito'])['y'].sum().reset_index()
monthly_total_all = monthly.groupby('periodo')['y'].sum().reset_index()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Grafica superior: Total mensual con area
xs = range(len(monthly_total_all))
ax1.fill_between(xs, monthly_total_all['y'].values, alpha=0.4, color='#457b9d')
ax1.plot(xs, monthly_total_all['y'].values, color='#8ecae6', linewidth=2)
ax1.set_xticks(xs[::3])
ax1.set_xticklabels(monthly_total_all['periodo'].values[::3], rotation=45, ha='right')
ax1.set_title('Volumen Total Mensual de Incidentes — La Paz 2023-2026')
ax1.set_ylabel('Incidentes/mes')
ax1.grid(True, alpha=0.3)

# Grafica inferior: Por zona (apilado)
bottom = None
periodos = monthly_total_all['periodo'].values
for i, zona in enumerate(zonas):
    vals = monthly_total[monthly_total['macrodistrito'] == zona].set_index('periodo')['y'].reindex(periodos).fillna(0).values
    ax2.bar(range(len(periodos)), vals, bottom=bottom,
            label=zona, color=PALETA_ZONAS[i], width=0.9)
    bottom = vals if bottom is None else bottom + vals

ax2.set_xticks(range(len(periodos))[::3])
ax2.set_xticklabels(periodos[::3], rotation=45, ha='right')
ax2.set_title('Incidentes Mensuales por Macrodistrito (Apilado)')
ax2.set_ylabel('Incidentes/mes')
ax2.legend(bbox_to_anchor=(1.01, 1), loc='upper left', framealpha=0.3)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '06_tendencia_anual.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Canal de Reporte y Patron Horario

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Canal de reporte
canal_counts = df_ind['canal_reporte'].value_counts()
axes[0].barh(canal_counts.index, canal_counts.values,
             color=sns.color_palette('viridis', len(canal_counts)))
axes[0].set_title('Distribucion por Canal de Reporte')
axes[0].set_xlabel('Cantidad de incidentes')
axes[0].grid(axis='x', alpha=0.3)
for i, (idx, val) in enumerate(canal_counts.items()):
    axes[0].text(val + 5, i, f'{val:,}', va='center', fontsize=9, color='white')

# Patron horario
hourly = df_ind['hour'].value_counts().sort_index()
colors_hour = ['#e63946' if (h >= 20 or h <= 4) else '#2a9d8f' for h in hourly.index]
axes[1].bar(hourly.index, hourly.values, color=colors_hour, width=0.9)
axes[1].set_title('Distribucion de Incidentes por Hora del Dia\n(Rojo = horas criticas nocturnas)')
axes[1].set_xlabel('Hora del dia (0-23)')
axes[1].set_ylabel('Cantidad de incidentes')
axes[1].set_xticks(range(0, 24, 2))
axes[1].grid(axis='y', alpha=0.3)
patch_noche = mpatches.Patch(color='#e63946', label='Horas criticas (20h-4h)')
patch_dia = mpatches.Patch(color='#2a9d8f', label='Horas diurnas')
axes[1].legend(handles=[patch_noche, patch_dia], framealpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, '07_canal_horario.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Correlacion y Calidad del Dataset

In [ ]:
print('=== REPORTE DE CALIDAD DEL DATASET ===')
print(f'  Total incidentes individuales : {len(df_ind):,}')
print(f'  Total registros diarios       : {len(df_daily):,}')
print(f'  Dias unicos cubiertos         : {df_daily["ds"].nunique()}')
print(f'  Macrodistritos                : {df_daily["macrodistrito"].nunique()}')
print(f'  Valores nulos (daily)         : {df_daily.isnull().sum().sum()}')
print(f'  Valores nulos (individual)    : {df_ind.isnull().sum().sum()}')
print(f'  Rango de fechas               : {df_daily["ds"].min()} -> {df_daily["ds"].max()}')
print()
print('=== DISTRIBUCION POR ESTADO ===')
print(df_ind['estado'].value_counts().to_string())
print()
print('[OK] EDA Completo. Siguiente paso: 02_baseline.ipynb')